Multi-label text classification model, for finding issues in business requirement

The model is able to detect 5 categories: Subjective, Ambiguous, Vague, Nonverifiable and Negative

The braking curves shall ensure that the train complies with its speed requirements.

Subjective - complies
Vague - its

In [25]:
# 1) Install / import
!pip install -q google-genai
from google import genai
from pydantic import BaseModel
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_KEY')
client = genai.Client(api_key=GEMINI_KEY)

MODEL_NAME = "gemini-2.5-flash"

class Scores(BaseModel):
    ambiguous: float
    subjective: float
    nonverifiable: float
    negative: float
    vague: float

def evaluate_requirement(requirement: str):
    prompt = f"""
Evaluate the following business requirement:

\"\"\"{requirement}\"\"\"\n

Return TWO SECTIONS:

<reasoning>
Explain briefly why each of the five dimensions receives its score.
</reasoning>

<scores>
Return JSON with float scores (0–1) for:
- ambiguous
- subjective
- nonverifiable
- negative
- vague
</scores>

Rules:
- If a label does not apply, assign 0.
- JSON goes ONLY inside <scores>.
"""
    # ✅ Minimal change: remove response_schema and mime type
    resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    # Now resp.text contains:
    # <reasoning> ... </reasoning>
    # <scores> { ... } </scores>
    return resp.text, None

# ---- Test ----
result, _ = evaluate_requirement(
    "The braking curves shall ensure that the train complies with its speed requirements."
)
print(result)

<reasoning>
**ambiguous**
The term "speed requirements" is not defined, leading to multiple possible interpretations of which specific speed limits, profiles, or operational constraints the braking curves must address.

**subjective**
Once the "speed requirements" are clearly defined and referenced, the compliance of the braking curves can be objectively measured through simulation, testing, or calculations, rather than relying on personal judgment or opinion.

**nonverifiable**
Without a clear definition or reference to specific "speed requirements" (e.g., maximum speed, sectional speed limits, emergency stopping distances), it is impossible to design tests or simulations to objectively verify that the braking curves ensure compliance.

**negative**
The requirement clearly states a desired positive outcome (ensuring compliance with speed requirements) rather than describing what the system should prevent or not do.

**vague**
The requirement lacks specific detail regarding the nature 

In [23]:
!pip install -q google-genai
from google import genai
from pydantic import BaseModel
GEMINI_KEY = userdata.get('GEMINI_KEY')

client = genai.Client(api_key=GEMINI_KEY)

# Pick a valid model printed by: for m in client.models.list(): print(m.name)
MODEL_NAME = "gemini-2.5-flash"  # change if you prefer "gemini-2.5-pro" etc.

# Scores schema (kept in case you want to parse JSON later manually)
class Scores(BaseModel):
    ambiguous: float
    subjective: float
    nonverifiable: float
    negative: float
    vague: float

# Few-shot examples
FEW_SHOTS = """
You are evaluating business requirements on five dimensions:
- ambiguous
- subjective
- nonverifiable
- negative
- vague
Each score is a float from 0 to 1. If a label does not apply, use 0.

Example 1
Requirement:
"The system shall be intuitive and self explanatory."
Return:
{"ambiguous": 0.013, "subjective": 0.997, "nonverifiable": 0.014, "negative": 0.009, "vague": 0.010}

Example 2
Requirement:
"In the presence of a maximum 2.5 second power-on skew, the FTPP system shall (3.1.6) be capable of completing FCC system power-up and initialization without synchronization errors."
Return:
{"ambiguous": 0.029, "subjective": 0.034, "nonverifiable": 0.976, "negative": 0.821, "vague": 0.035}
"""

def evaluate_requirement(requirement: str):
    prompt = f"""{FEW_SHOTS}

Evaluate the following requirement and return two sections:

<reasoning>
Explain briefly why each dimension receives its score.
</reasoning>

<scores>
Return JSON with floats between 0 and 1 for: ambiguous, subjective, nonverifiable, negative, vague.
</scores>

Requirement:
\"\"\"{requirement}\"\"\""""

    # minimal change: no schema, free-form output
    resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return resp.text  # full reasoning + json block

# Test
result = evaluate_requirement("The braking curves shall ensure that the train complies with its speed requirements.")
print(result)

<reasoning>
**ambiguous**: The term "speed requirements" is highly ambiguous. It could refer to various aspects such as maximum speed limits, safe speeds for curves, specific stopping distances, or adherence to a timetable. This lack of clarity means the requirement can be interpreted in multiple ways, and the scope of "ensure" is also not clearly defined.
**subjective**: The requirement deals with objective physical properties (braking curves, train speed, and compliance). While the terms are poorly defined, the underlying concepts are not dependent on personal opinion or judgment. The goal is to meet measurable (albeit currently unspecified) criteria.
**nonverifiable**: Without a clear, quantifiable, and measurable definition of "speed requirements" (e.g., specific speed values, distances, or conditions), it is impossible to objectively test, simulate, or demonstrate that the braking curves successfully "ensure" compliance. The requirement is untestable as written.
**negative**: The 

Below is the Masters model

In [20]:
!pip install -q setfit datasets sentence-transformers scikit-learn huggingface_hub

from setfit import SetFitModel
from huggingface_hub import hf_hub_download
import json, numpy as np

MODEL_ID = "Hulyyy/req-quality-setfit-2"
HF_API_KEY = userdata.get('HF_API_KEY')

# 1) Load model
model = SetFitModel.from_pretrained(MODEL_ID, token=HF_API_KEY)

# 2) Load config (supports both names)
cfg_path = None
for fname in ("config_labels.json", "config_label.json"):
    try:
        cfg_path = hf_hub_download(repo_id=MODEL_ID, filename=fname)
        break
    except Exception:
        pass
if cfg_path is None:
    raise FileNotFoundError("Neither config_labels.json nor config_label.json found in the repo.")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

# Parse labels
id2label = {int(k): v for k, v in cfg["id2label"].items()}
num_labels = len(id2label)

# 3) Load thresholds if provided, else default 0.5
thresholds = np.full(num_labels, 0.5, dtype=float)
thr_file = cfg.get("label_thresholds.npy")
if thr_file:
    try:
        thr_path = hf_hub_download(repo_id=MODEL_ID, filename=thr_file)
        thresholds = np.load(thr_path)
    except Exception:
        # keep default 0.5 if missing
        pass

# 4) Predict (probabilities -> threshold -> label names)
texts = ["The braking curves shall ensure that the train complies with its speed requirements."]
probas = np.array(model.predict_proba(texts))             # shape (N, L)
preds = (probas >= thresholds.reshape(1, -1)).astype(int) # 0/1 per label
decoded = [[id2label[i] for i, v in enumerate(row) if v == 1] for row in preds]

# 5) Print
for t, labels, p in zip(texts, decoded, probas):
    print("\nText:", t)
    print("Predicted labels:", labels)
    print("Probabilities:", np.round(p, 3).tolist())

config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

...

model_head.pkl:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ClassifierChain from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


config_labels.json:   0%|          | 0.00/428 [00:00<?, ?B/s]


Text: The braking curves shall ensure that the train complies with its speed requirements.
Predicted labels: ['Subjective', 'Vague']
Probabilities: [0.87, 0.01, 0.03, 0.033, 0.937]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


SetFit-based solution uses a domain-trained multi-label classifier specifically optimized for requirement quality analysis. It loads a fine-tuned SetFit model, reads label definitions and thresholds, computes probability scores, and applies those thresholds to produce concise, discriminative label predictions. For the evaluated requirement, your model confidently activates only Nonverifiable and Negative, with high probabilities that reflect clear learned separation between relevant and irrelevant labels.

Gemini’s zero-shot approach, in contrast, provides soft continuous scores without domain specialization. Its outputs spread moderately across multiple categories, capturing some signals but lacking the sharp decision boundaries of your SetFit model. Because no thresholds are applied and no task-specific training is involved, the zero-shot predictions are more diffuse and less aligned with your classifier’s behavior.

When few-shot examples are added to Gemini, the model becomes heavily influenced by the examples provided. Since the examples include high probability values across most dimensions, the model generalizes this pattern and assigns uniformly high scores to all categories. This over-conditioning effect is common in LLMs: they mirror the structure and magnitude of the examples rather than independently assessing the input. As a result, the few-shot version becomes overly sensitive, producing inflated scores that diverge significantly from your classifier’s more precise outputs.

Overall, the SetFit model remains the most accurate and selective because it is trained directly on the requirement-quality task. Gemini zero-shot offers general-purpose scoring but lacks granularity, while Gemini few-shot adapts too strongly to the example format and requires more balanced demonstrations to behave consistently.